# Day 7—Seeing the pattern
*Measuring Manuscripts*

You can't see fifty dimensions at once. Dimensionality reduction compresses high-dimensional data—sign shapes, vowels, word counts—to two, so the clusters become visible. The compression distorts, so we read the picture and then question it.

## 1. Setup

In [ ]:
!pip install umap-learn scikit-learn --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Data with real clusters

150 points in 20 dimensions, in three genuine clusters. Treat each point as a sign, a vowel, or a text. Your real features go in at the end.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=150, centers=3, n_features=20, cluster_std=2.5, random_state=0)
# === TO BUILD: replace X (features) and y (labels, e.g. period/hand) with real data ===
print('data:', X.shape)

## 3. PCA: the linear view

Fast, with interpretable axes. It can blur fine structure, but it's hard to mislead with.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
p = pca.fit_transform(X)
plt.figure(figsize=(6, 5))
plt.scatter(p[:, 0], p[:, 1], c=y, cmap='tab10', s=18)
plt.title('PCA'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

## 4. How much did PCA keep?

Those two axes are a summary, and a summary omits things. The explained-variance ratio says how much of the original structure survived the compression.

In [ ]:
full = PCA().fit(X)
plt.figure(figsize=(6, 3))
plt.bar(range(1, 11), full.explained_variance_ratio_[:10])
plt.xlabel('component'); plt.ylabel('variance explained')
plt.title('First two bars are all you see in the scatter'); plt.show()
print('PC1+PC2 keep', round(100 * full.explained_variance_ratio_[:2].sum()), '% of the variance.')

## 4b. What do the PCA axes mean?

Each PCA axis is a fixed recipe of your original features. Read the recipe and you can sometimes name the axis. Here are the features that weigh most heavily on PC1 and PC2—the ones that decide left–right and up–down in the scatter.

In [ ]:
# How each original feature contributes to PC1 and PC2 (the 'loadings')
loadings = pca.components_           # shape: 2 axes x 20 features
for axis, name in [(0, 'PC1'), (1, 'PC2')]:
    order = np.argsort(np.abs(loadings[axis]))[::-1][:3]
    top = ', '.join(f'feature {j} ({loadings[axis][j]:+.2f})' for j in order)
    print(f'{name} is driven most by:', top)
# === TO BUILD: with real Day 4 sign features, swap 'feature j' for names
#     like height, width, loop_area so the axes can be interpreted ===

## 5. UMAP: separating clusters

Nonlinear, and good at pulling clusters apart. But the distances between clusters can be unreliable, so don't read meaning into the gaps.

In [ ]:
import umap
u = umap.UMAP(random_state=0).fit_transform(X)
plt.figure(figsize=(6, 5))
plt.scatter(u[:, 0], u[:, 1], c=y, cmap='tab10', s=18)
plt.title('UMAP'); plt.show()

## 6. UMAP is sensitive: vary one setting

`n_neighbors` controls how local or global UMAP's view is. Change it and the picture changes. None is the true one; they answer slightly different questions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, nn in zip(axes, [5, 15, 50]):
    emb = umap.UMAP(n_neighbors=nn, random_state=0).fit_transform(X)
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=12)
    ax.set_title(f'n_neighbors = {nn}')
plt.show()

## 7. t-SNE, for comparison

Another nonlinear method. Compare its picture with UMAP's; where they agree, the structure is more trustworthy.

In [ ]:
from sklearn.manifold import TSNE
ts = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X)
plt.figure(figsize=(6, 5))
plt.scatter(ts[:, 0], ts[:, 1], c=y, cmap='tab10', s=18)
plt.title('t-SNE'); plt.show()

## 7b. t-SNE is sensitive too: vary perplexity

`perplexity` is t-SNE's main knob—roughly how many neighbors each point tries to stay close to. Low values stress small local clumps; high values stress broader groups. As with UMAP's `n_neighbors`, no single value is correct.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, perp in zip(axes, [5, 30, 50]):
    emb = TSNE(n_components=2, perplexity=perp, random_state=0).fit_transform(X)
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=12)
    ax.set_title(f'perplexity = {perp}')
plt.show()

## 8. Put a number on the clusters

Eyeballing clusters is subjective. Cluster the embedding automatically and score it against the true labels; a score near 1 means the clustering recovered the real groups.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
pred = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(u)
print('Adjusted Rand score vs true labels:', round(adjusted_rand_score(y, pred), 2))

## 8b. Does the projection help the clustering?

Cluster the raw 20-dimensional data and the 2-D UMAP embedding, and score each against the true labels. If the embedding scores at least as well, the projection kept the structure that matters.

In [ ]:
km_raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
km_emb = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(u)
print('Adjusted Rand, raw 20-D data:', round(adjusted_rand_score(y, km_raw), 2))
print('Adjusted Rand, 2-D UMAP     :', round(adjusted_rand_score(y, km_emb), 2))
# 1.0 = perfect recovery of the true groups; 0.0 = no better than chance

## 8c. Hierarchical clustering: let the data pick the number

k-means needs you to name the number of groups. Hierarchical clustering does not: it merges the two closest groups over and over and records the order as a tree. Long vertical jumps in the tree mark natural cut points—where the data resists merging.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
Z = linkage(X, method='ward')
plt.figure(figsize=(9, 3))
dendrogram(Z, no_labels=True)
plt.title('Cut where the vertical jumps are longest'); plt.ylabel('distance'); plt.show()

## 9. Two seeds, two pictures

Run UMAP twice with different random seeds. The clusters are the same, but the spaces between them—the part the eye reads as 'how different'—are not. Don't build an argument on the gaps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, seed in zip(axes, [1, 2]):
    emb = umap.UMAP(random_state=seed).fit_transform(X)
    ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap='tab10', s=12)
    ax.set_title(f'random_state = {seed}')
plt.show()

## 9b. Spotting an outlier

A point far from every cluster centre is a candidate oddity—a rare form, or a measurement error. Find it by distance, then go back to the source image or recording to decide which it is. The plot tells you where to look, not what you found.

In [ ]:
from scipy.spatial.distance import cdist
centres = np.array([X[y == g].mean(axis=0) for g in np.unique(y)])
dist_to_nearest = cdist(X, centres).min(axis=1)
worst = np.argsort(dist_to_nearest)[::-1][:3]
print('Most outlying points (index, distance):')
for i in worst:
    print(f'  point {i}: {dist_to_nearest[i]:.1f}')

plt.figure(figsize=(6, 5))
plt.scatter(u[:, 0], u[:, 1], c='lightgrey', s=18)
plt.scatter(u[worst, 0], u[worst, 1], c='red', s=80, marker='x')
plt.title('Three most outlying points (UMAP)'); plt.show()
# === TO BUILD: trace each flagged point back to its source: real rare form,
#     or a digitization/segmentation error? ===

## 10. Your real features

> 🔧 **TO BUILD**—point `X` and `y` at your own data: sign-shape features from Day 4, or vowel formants from Day 5. The same four pictures then describe hands, periods, or dialects.

### Talk it over
- Did the clusters survive PCA, UMAP, and t-SNE? Which view do you trust most?
- Pick an outlier. Real exception, or a digitization error, and how would you check?
- **Check-in:** what single image would make your finding clear at a glance?